# 13 — Supplemental JPEG + SigLIP producer (GPU, preflight → shard)

Notebook này đọc freeze của notebook 12 và tạo **research artifacts**, không
nhập BTC/live index. Nó xử lý union 17.399 frame một lần dù frame có thể thuộc
nhiều arm. Mỗi video tạo deterministic JPEG tar, image manifest, SigLIP NPZ,
report và COMPLETE marker.

Trước khi mở shard, notebook lấy một BTC JPEG thật cùng vector SigLIP đã lưu,
chạy lại pipeline và đo numeric canary. `PREFLIGHT` chỉ chạy một video ngắn nhất
của worker. Chỉ đổi sang `SHARD` khi canary + environment PASS.

Đây mới là **SigLIP parity**. Full feature parity vẫn BLOCKED cho tới khi OCR,
caption/BGE và downstream fingerprint được tạo/đối chiếu. Không được gọi output
này production-ready hoặc official-score uplift.

### Cách dùng

1. Chọn GPU, bật Internet; secrets GitHub + Modal như notebook trước.
2. Đổi đúng `MEMBER`, `WORKER`; để `RUN_SCOPE="PREFLIGHT"`, Run All.
3. Gửi lead output `SIGLIP_PREFLIGHT`.
4. Chỉ khi PASS mới đặt `RUN_SCOPE="SHARD"` và `CONFIRM_SHARD=WORKER`.
5. Rerun tự bỏ video có COMPLETE marker hợp lệ.

## 0 — Chỉ sửa MEMBER / WORKER / RUN_SCOPE

In [ ]:
from pathlib import Path

MEMBER = "quang"
WORKER = "quang1"
RUN_SCOPE = "PREFLIGHT"             # PREFLIGHT hoặc SHARD
CONFIRM_SHARD = ""                  # SHARD yêu cầu đúng WORKER
SESSION_MAX_MINUTES = 240
SIGLIP_BATCH_SIZE = 32
JPEG_QUALITY = 95
CANARY_TOLERANCE = 1e-5

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/btc-delta-siglip")

EXPECTED_SELECTION_FREEZE_SHA256 = "75296b93ab4428ae721776d90167cdf3bca6b262137fd93610d04082def5a630"
EXPECTED_CORPUS_LOCK_HASH = "2e0887652464e0cc7942bedb94177cfe2dff5f7baf77081902461cf3c885d75f"
SELECTION_SUBDIR = "gate1-selection-v1"
FEATURE_SUBDIR = "gate1-features-siglip-v1"
CHECKPOINT = "google/siglip-so400m-patch14-384"
CANARY_VIDEO_ID = "L21_V001"

if RUN_SCOPE not in {"PREFLIGHT", "SHARD"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT hoặc SHARD")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != WORKER:
    raise RuntimeError("SHARD bị khóa; đặt CONFIRM_SHARD đúng bằng WORKER sau preflight PASS")
if not 1 <= SIGLIP_BATCH_SIZE <= 64:
    raise ValueError("SIGLIP_BATCH_SIZE phải trong [1,64]")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 1 — Refresh repo, dependency và frozen selection

In [ ]:
import csv
import gc
import importlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:3])}\n{result.stderr[-1500:]}")
    return result.stdout.strip()


def refresh_repo() -> None:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    try:
        if (REPO_DIR / ".git").is_dir():
            if run(["git", "status", "--porcelain"], cwd=REPO_DIR):
                raise RuntimeError("Checkout Kaggle có local changes; Restart Session rồi Run All")
            run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO_DIR, env=env)
            run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
        else:
            run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], env=env)
    finally:
        askpass.unlink(missing_ok=True)


refresh_repo()
requirements = REPO_DIR / "notebooks/requirements-kaggle-btc-images.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
sys.path.insert(0, str(REPO_DIR / "src"))

import numpy as np
import torch
import yaml
from mfe.contracts.ids import build_keyframe_id
from mfe.offline.bundle import BundleLayout, check_vectors, freeze_environment
from mfe.offline.keyframe_augmentation.common import atomic_write_json, read_jsonl, sha256_file, sha256_json
from mfe.offline.keyframe_augmentation.corpus_audit import expected_worker_assignments
from mfe.offline.keyframe_augmentation.delta_siglip import image_manifest, partition_feature_worklist, write_deterministic_image_tar
from mfe.offline.source import fetch_video, load_catalogue
from mfe.offline.stages import _siglip_feature_tensor
from mfe.offline.transport import ModalVolumeTransport

if not torch.cuda.is_available():
    raise RuntimeError("Không thấy CUDA; chọn GPU accelerator rồi Restart Session")

REPO_COMMIT = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
PRODUCER_CONFIG_PATH = REPO_DIR / "config/keyframe_augmentation_kaggle.yaml"
PRODUCER_CONFIG = yaml.safe_load(PRODUCER_CONFIG_PATH.read_text(encoding="utf-8"))
INVENTORY_PATH = REPO_DIR / PRODUCER_CONFIG["source"]["catalogue"]
INVENTORY = yaml.safe_load(INVENTORY_PATH.read_text(encoding="utf-8"))
CATALOGUE = load_catalogue(INVENTORY_PATH)
EXPECTED = expected_worker_assignments(producer_config=PRODUCER_CONFIG, inventory=INVENTORY, catalogue=CATALOGUE)
if WORKER not in EXPECTED or not WORKER.startswith(MEMBER):
    raise ValueError(f"MEMBER/WORKER không hợp lệ: {MEMBER}/{WORKER}")

BTC = ModalVolumeTransport(PRODUCER_CONFIG["btc"]["volume"], create_if_missing=False)
RESEARCH = ModalVolumeTransport(PRODUCER_CONFIG["output"]["volume"], create_if_missing=False)
BTC_LAYOUT = BundleLayout(PRODUCER_CONFIG["btc"]["bundle"])
PRODUCER_ROOT = PRODUCER_CONFIG["output"]["root"].rstrip("/")
SELECTION_ROOT = f"{PRODUCER_ROOT}/{SELECTION_SUBDIR}"
FEATURE_ROOT = f"{PRODUCER_ROOT}/{FEATURE_SUBDIR}"

INPUT_DIR = WORK_DIR / "input"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
FREEZE_PATH = INPUT_DIR / "selection-freeze.json"
RESEARCH.get(f"{SELECTION_ROOT}/selection-freeze.json", FREEZE_PATH)
if sha256_file(FREEZE_PATH) != EXPECTED_SELECTION_FREEZE_SHA256:
    raise RuntimeError("Selection freeze SHA drift")
SELECTION_FREEZE = json.loads(FREEZE_PATH.read_text(encoding="utf-8"))
unsigned_freeze = dict(SELECTION_FREEZE)
freeze_payload_hash = unsigned_freeze.pop("freeze_payload_hash")
if sha256_json(unsigned_freeze) != freeze_payload_hash or SELECTION_FREEZE["input_corpus_lock_hash"] != EXPECTED_CORPUS_LOCK_HASH:
    raise RuntimeError("Selection freeze payload/corpus lock không hợp lệ")

WORKLIST_PATH = INPUT_DIR / "feature-worklist.jsonl"
RESEARCH.get(f"{SELECTION_ROOT}/feature-worklist.jsonl", WORKLIST_PATH)
if sha256_file(WORKLIST_PATH) != SELECTION_FREEZE["files"]["feature-worklist.jsonl"]:
    raise RuntimeError("Feature worklist hash mismatch")
WORKLIST = list(read_jsonl(WORKLIST_PATH))
PARTITIONS = partition_feature_worklist(WORKLIST, EXPECTED)
ASSIGNED_ROWS = PARTITIONS[WORKER]
BY_VIDEO = defaultdict(list)
for row in ASSIGNED_ROWS:
    BY_VIDEO[row["video_id"]].append(row)
print({"repo_commit": REPO_COMMIT, "gpu": torch.cuda.get_device_name(0), "member": MEMBER, "worker": WORKER, "scope": RUN_SCOPE, "assigned_videos": len(BY_VIDEO), "assigned_frames": len(ASSIGNED_ROWS), "selection_freeze_sha256": EXPECTED_SELECTION_FREEZE_SHA256})

## 2 — Numeric canary trên BTC JPEG + stored SigLIP vector

In [ ]:
from huggingface_hub import model_info, snapshot_download
from PIL import Image
from transformers import AutoModel, AutoProcessor

MODEL_REVISION = model_info(CHECKPOINT).sha
SNAPSHOT_DIR = Path(snapshot_download(CHECKPOINT, revision=MODEL_REVISION))
MODEL_FILE_HASHES = {
    str(path.relative_to(SNAPSHOT_DIR)): sha256_file(path)
    for path in sorted(SNAPSHOT_DIR.rglob("*"))
    if path.is_file() and (path.suffix in {".json", ".safetensors", ".txt"} or path.name.startswith("tokenizer"))
}
MODEL_SNAPSHOT_HASH = sha256_json(MODEL_FILE_HASHES)
PROCESSOR = AutoProcessor.from_pretrained(CHECKPOINT, revision=MODEL_REVISION)
MODEL = AutoModel.from_pretrained(CHECKPOINT, revision=MODEL_REVISION).eval().cuda()

CANARY_DIR = WORK_DIR / "canary"
CANARY_DIR.mkdir(parents=True, exist_ok=True)
baseline_manifest_path = CANARY_DIR / "part-manifest.json"
baseline_vectors_path = CANARY_DIR / "siglip.npz"
baseline_mapping_path = CANARY_DIR / "mapping.csv"
BTC.get(BTC_LAYOUT.part_manifest(CANARY_VIDEO_ID), baseline_manifest_path)
BTC.get(BTC_LAYOUT.part("siglip_vectors", CANARY_VIDEO_ID), baseline_vectors_path)
BTC.get(BTC_LAYOUT.map_keyframes(CANARY_VIDEO_ID), baseline_mapping_path)
BASELINE_MANIFEST = json.loads(baseline_manifest_path.read_text(encoding="utf-8"))
BASELINE_ENV = BASELINE_MANIFEST["stages"]["n4_siglip"]["env"]
CURRENT_ENV = freeze_environment()
ENV_FIELDS = ("torch", "transformers", "numpy")
ENV_MISMATCHES = {
    field: {"btc": BASELINE_ENV.get(field), "current": CURRENT_ENV.get(field)}
    for field in ENV_FIELDS if BASELINE_ENV.get(field) != CURRENT_ENV.get(field)
}

stored = np.load(baseline_vectors_path, allow_pickle=False)
stored_ids = stored["ids"].astype(str).tolist()
stored_by_id = {value: index for index, value in enumerate(stored_ids)}
with baseline_mapping_path.open("r", encoding="utf-8-sig", newline="") as stream:
    mapping_rows = list(csv.DictReader(stream))
canary_row = next(row for row in mapping_rows if build_keyframe_id(CANARY_VIDEO_ID, int(row["frame_idx"])) in stored_by_id)
CANARY_ID = build_keyframe_id(CANARY_VIDEO_ID, int(canary_row["frame_idx"]))
canary_image_path = CANARY_DIR / "btc-canary.jpg"
BTC.get(f"{BTC_LAYOUT.keyframes_dir(CANARY_VIDEO_ID)}/{int(canary_row['n']):03d}.jpg", canary_image_path)

inputs = PROCESSOR(images=[Image.open(canary_image_path).convert("RGB")], return_tensors="pt").to("cuda")
with torch.no_grad():
    raw = _siglip_feature_tensor(MODEL.get_image_features(**inputs)).float().cpu().numpy()
checked = check_vectors([CANARY_ID], raw, CANARY_VIDEO_ID, 1152)
current_vector = checked.kept[0]
baseline_vector = stored["vectors"][stored_by_id[CANARY_ID]].astype(np.float32)
CANARY_MAX_ABS_DIFF = float(np.max(np.abs(current_vector - baseline_vector)))
CANARY_COSINE = float(np.dot(current_vector, baseline_vector) / (np.linalg.norm(current_vector) * np.linalg.norm(baseline_vector)))
CANARY_PASS = CANARY_MAX_ABS_DIFF <= CANARY_TOLERANCE
ENV_PASS = not ENV_MISMATCHES
PREFLIGHT_PASS = CANARY_PASS and ENV_PASS
CANARY_REPORT = {
    "schema_version": 1,
    "status": "PASS" if PREFLIGHT_PASS else "BLOCKED",
    "canary_id": CANARY_ID,
    "btc_jpeg_sha256": sha256_file(canary_image_path),
    "btc_siglip_part_sha256": sha256_file(baseline_vectors_path),
    "sample_count": 1,
    "max_abs_diff": CANARY_MAX_ABS_DIFF,
    "cosine": CANARY_COSINE,
    "tolerance": CANARY_TOLERANCE,
    "environment_mismatches": ENV_MISMATCHES,
    "model": CHECKPOINT,
    "resolved_revision": MODEL_REVISION,
    "model_snapshot_hash": MODEL_SNAPSHOT_HASH,
    "full_feature_parity_status": "BLOCKED_MISSING_OCR_CAPTION_BGE_AND_DOWNSTREAM",
}
CANARY_REPORT["report_hash"] = sha256_json(CANARY_REPORT)
atomic_write_json(CANARY_DIR / "siglip-canary-report.json", CANARY_REPORT)
print({"SIGLIP_PREFLIGHT": CANARY_REPORT})
if not PREFLIGHT_PASS:
    raise RuntimeError("SigLIP preflight BLOCKED; không chạy candidate shard")

## 3 — Resume inventory và chọn todo

In [ ]:
def remote_paths(video_id: str) -> dict[str, str]:
    return {
        "images": f"{FEATURE_ROOT}/images/{video_id}.tar",
        "image_manifest": f"{FEATURE_ROOT}/image-manifests/{video_id}.json",
        "siglip": f"{FEATURE_ROOT}/siglip/{video_id}.npz",
        "report": f"{FEATURE_ROOT}/reports/{video_id}.json",
        "marker": f"{FEATURE_ROOT}/markers/{MEMBER}/{WORKER}/{video_id}.json",
    }


def completion_marker(video_id: str) -> dict | None:
    remote = remote_paths(video_id)
    if not RESEARCH.exists(remote["marker"]):
        return None
    local = WORK_DIR / "marker-check" / f"{video_id}.json"
    try:
        RESEARCH.get(remote["marker"], local)
        marker = json.loads(local.read_text(encoding="utf-8"))
        unsigned = dict(marker)
        payload_hash = unsigned.pop("marker_payload_hash", None)
        if payload_hash != sha256_json(unsigned):
            return None
        if marker.get("status") != "COMPLETE" or marker.get("selection_freeze_sha256") != EXPECTED_SELECTION_FREEZE_SHA256:
            return None
        if marker.get("worker") != WORKER or marker.get("video_id") != video_id:
            return None
        for key in ("images", "image_manifest", "siglip", "report"):
            if RESEARCH.sha256(remote[key]) != marker.get(f"{key}_sha256"):
                return None
        return marker
    except Exception:
        return None
    finally:
        local.unlink(missing_ok=True)


completed = [video_id for video_id in sorted(BY_VIDEO) if completion_marker(video_id)]
pending = [video_id for video_id in sorted(BY_VIDEO) if video_id not in set(completed)]
pending_by_size = sorted(pending, key=lambda video_id: (CATALOGUE[video_id].size, video_id))
TODO = pending_by_size[:1] if RUN_SCOPE == "PREFLIGHT" else pending_by_size
print({"assigned_videos": len(BY_VIDEO), "complete": len(completed), "pending": len(pending), "todo_this_run": len(TODO), "first_todo": TODO[:3]})

## 4 — Decode exact frame, JPEG → SigLIP, upload + marker

In [ ]:
import cv2

ATTACHED = {}
for path in Path("/kaggle/input").rglob("*.mp4") if Path("/kaggle/input").is_dir() else []:
    ATTACHED.setdefault(path.stem, []).append(path)


def resolve_video(video_id: str, destination: Path) -> tuple[Path, str]:
    expected_size = CATALOGUE[video_id].size
    exact = [path for path in ATTACHED.get(video_id, []) if path.stat().st_size == expected_size]
    if len(exact) == 1:
        return exact[0], "kaggle-input-read-only"
    if len(exact) > 1 or ATTACHED.get(video_id):
        raise RuntimeError(f"Attached MP4 ambiguous/wrong size: {video_id}")
    fetch_video(CATALOGUE[video_id], destination)
    return destination, "http-range"


def decode_rows(video_path: Path, rows: list[dict], output_dir: Path) -> dict:
    targets = {int(row["frame_idx"]) for row in rows}
    output_dir.mkdir(parents=True, exist_ok=True)
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Không mở được video: {video_path}")
    decoder_fps = float(capture.get(cv2.CAP_PROP_FPS))
    found = set()
    frame_idx = 0
    try:
        while targets - found:
            ok, bgr = capture.read()
            if not ok:
                break
            if frame_idx in targets:
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                Image.fromarray(rgb).save(output_dir / f"{frame_idx:09d}.jpg", format="JPEG", quality=JPEG_QUALITY, subsampling=0)
                found.add(frame_idx)
            frame_idx += 1
    finally:
        capture.release()
    missing = sorted(targets - found)
    if missing:
        raise RuntimeError(f"Decode thiếu frame {missing[:5]} / {len(missing)}")
    literal_fps = {float(row["fps"]) for row in rows}
    if len(literal_fps) != 1 or abs(decoder_fps - next(iter(literal_fps))) > 0.05:
        raise RuntimeError(f"Decoder/literal FPS mismatch: {decoder_fps} vs {literal_fps}")
    return {"decoder": "opencv.VideoCapture sequential integer counter", "opencv": cv2.__version__, "decoder_fps": decoder_fps, "decoded_frames": len(found)}


def siglip_vectors(rows: list[dict], image_dir: Path) -> tuple[list[str], np.ndarray]:
    ids = [str(row["keyframe_id"]) for row in sorted(rows, key=lambda value: int(value["frame_idx"]))]
    paths = [image_dir / f"{int(row['frame_idx']):09d}.jpg" for row in sorted(rows, key=lambda value: int(value["frame_idx"]))]
    vectors = []
    for start in range(0, len(paths), SIGLIP_BATCH_SIZE):
        images = [Image.open(path).convert("RGB") for path in paths[start:start + SIGLIP_BATCH_SIZE]]
        inputs = PROCESSOR(images=images, return_tensors="pt").to("cuda")
        with torch.no_grad():
            batch = _siglip_feature_tensor(MODEL.get_image_features(**inputs))
        vectors.append(batch.float().cpu().numpy())
    checked = check_vectors(ids, np.concatenate(vectors), rows[0]["video_id"], 1152)
    if checked.dropped or len(checked.kept_ids) != len(ids):
        raise RuntimeError("SigLIP produced invalid/zero vectors")
    return checked.kept_ids, checked.kept


def process_video(video_id: str) -> dict:
    started = time.perf_counter()
    rows = BY_VIDEO[video_id]
    local = WORK_DIR / "videos" / video_id
    image_dir = local / "images"
    local.mkdir(parents=True, exist_ok=True)
    video_local = local / f"{video_id}.mp4"
    try:
        print(f"{video_id}: fetch + decode {len(rows)} frames", flush=True)
        video_path, source_mode = resolve_video(video_id, video_local)
        video_sha = sha256_file(video_path)
        decode_report = decode_rows(video_path, rows, image_dir)
        manifest = image_manifest(video_id=video_id, rows=rows, image_dir=image_dir)
        manifest_path = local / "image-manifest.json"
        atomic_write_json(manifest_path, manifest)
        tar_path = write_deterministic_image_tar(
            {path.name: path for path in image_dir.glob("*.jpg")}, local / "images.tar"
        )
        print(f"{video_id}: SigLIP {len(rows)} frames", flush=True)
        ids, vectors = siglip_vectors(rows, image_dir)
        siglip_path = local / "siglip.npz"
        np.savez_compressed(siglip_path, ids=np.asarray(ids, dtype=np.str_), vectors=vectors)
        report = {
            "schema_version": 1, "video_id": video_id, "member": MEMBER, "worker": WORKER,
            "selection_freeze_sha256": EXPECTED_SELECTION_FREEZE_SHA256,
            "source_mode": source_mode, "source_video_sha256": video_sha,
            "decode": decode_report, "jpeg": {"library": "Pillow", "color": "RGB", "quality": JPEG_QUALITY, "subsampling": 0},
            "siglip": {"model": CHECKPOINT, "revision": MODEL_REVISION, "snapshot_hash": MODEL_SNAPSHOT_HASH, "batch_size": SIGLIP_BATCH_SIZE, "dtype": "float32", "dim": 1152, "embedding_norm": "L2"},
            "siglip_canary_report_hash": CANARY_REPORT["report_hash"],
            "siglip_parity_status": "MEASURED_NUMERIC_CANARY",
            "full_feature_parity_status": "BLOCKED_MISSING_OCR_CAPTION_BGE_AND_DOWNSTREAM",
            "repo_commit": REPO_COMMIT, "elapsed_seconds": round(time.perf_counter() - started, 3),
        }
        report["report_payload_hash"] = sha256_json(report)
        report_path = local / "report.json"
        atomic_write_json(report_path, report)
        paths = {"images": tar_path, "image_manifest": manifest_path, "siglip": siglip_path, "report": report_path}
        hashes = {key: sha256_file(path) for key, path in paths.items()}
        remote = remote_paths(video_id)
        print(f"{video_id}: upload + verify", flush=True)
        for key, path in paths.items():
            RESEARCH.put(path, remote[key])
            if RESEARCH.sha256(remote[key]) != hashes[key]:
                raise RuntimeError(f"Remote verify failed: {video_id}/{key}")
        marker = {
            "schema_version": 1, "status": "COMPLETE", "video_id": video_id,
            "member": MEMBER, "worker": WORKER, "run_scope": RUN_SCOPE,
            "selection_freeze_sha256": EXPECTED_SELECTION_FREEZE_SHA256,
            "work_items_hash": sha256_json([row["work_item_hash"] for row in rows]),
            "frame_count": len(rows), "repo_commit": REPO_COMMIT,
            "model_snapshot_hash": MODEL_SNAPSHOT_HASH,
            **{f"{key}_sha256": value for key, value in hashes.items()},
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        marker["marker_payload_hash"] = sha256_json(marker)
        marker_path = local / "marker.json"
        atomic_write_json(marker_path, marker)
        RESEARCH.put(marker_path, remote["marker"])
        if RESEARCH.sha256(remote["marker"]) != sha256_file(marker_path):
            raise RuntimeError("Remote marker verify failed")
        return marker
    finally:
        shutil.rmtree(local, ignore_errors=True)
        gc.collect()
        torch.cuda.empty_cache()


session_started = time.perf_counter()
successes = []
failures = []
for ordinal, video_id in enumerate(TODO, 1):
    if ordinal > 1 and time.perf_counter() - session_started >= SESSION_MAX_MINUTES * 60:
        print({"session_checkpoint": "TIME_BUDGET_REACHED", "remaining": len(TODO) - ordinal + 1})
        break
    print(f"[{ordinal}/{len(TODO)}] {video_id} ({CATALOGUE[video_id].size / 2**20:.1f} MiB)")
    try:
        marker = process_video(video_id)
        successes.append(video_id)
        print({"complete": video_id, "frames": marker["frame_count"]})
    except Exception as error:
        failure = {"video_id": video_id, "error_type": type(error).__name__, "error": str(error), "at_utc": datetime.now(timezone.utc).isoformat()}
        failures.append(failure)
        print({"failed": failure})
        if RUN_SCOPE == "PREFLIGHT":
            raise

## 5 — Worker manifest và trạng thái bàn giao

In [ ]:
verified = []
for video_id in sorted(BY_VIDEO):
    marker = completion_marker(video_id)
    if marker:
        verified.append({"video_id": video_id, "frame_count": marker["frame_count"], "marker_payload_hash": marker["marker_payload_hash"], "siglip_sha256": marker["siglip_sha256"], "images_sha256": marker["images_sha256"]})
worker_manifest = {
    "schema_version": 1, "status": "COMPLETE" if len(verified) == len(BY_VIDEO) else "PARTIAL",
    "member": MEMBER, "worker": WORKER, "selection_freeze_sha256": EXPECTED_SELECTION_FREEZE_SHA256,
    "assigned_videos": len(BY_VIDEO), "assigned_frames": len(ASSIGNED_ROWS),
    "verified_videos": len(verified), "verified_frames": sum(row["frame_count"] for row in verified),
    "verified": verified, "failures_this_session": failures,
    "siglip_canary_report": CANARY_REPORT,
    "full_feature_parity_status": "BLOCKED_MISSING_OCR_CAPTION_BGE_AND_DOWNSTREAM",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
worker_manifest["manifest_payload_hash"] = sha256_json(worker_manifest)
manifest_path = WORK_DIR / f"worker-manifest-{WORKER}.json"
atomic_write_json(manifest_path, worker_manifest)
manifest_remote = f"{FEATURE_ROOT}/worker-manifests/{WORKER}.json"
RESEARCH.put(manifest_path, manifest_remote)
if RESEARCH.sha256(manifest_remote) != sha256_file(manifest_path):
    raise RuntimeError("Worker manifest remote verify failed")
print({"SIGLIP_PREFLIGHT": "PASS", "scope": RUN_SCOPE, "worker": WORKER, "verified_videos": len(verified), "assigned_videos": len(BY_VIDEO), "verified_frames": worker_manifest["verified_frames"], "assigned_frames": len(ASSIGNED_ROWS), "manifest_sha256": sha256_file(manifest_path), "full_feature_parity": worker_manifest["full_feature_parity_status"]})

## 6 — Diễn giải

- `PREFLIGHT PASS` chỉ cho phép mở shard SigLIP/JPEG.
- Worker chỉ xong khi `verified_videos == assigned_videos` và frame count khớp.
- Sau đủ 12 worker, lead phải hợp nhất manifest, audit image hash/vector shape,
  chạy pHash/embedding near-duplicate và tiếp tục OCR/caption/BGE parity.
- Không build live index và không dùng held-out ở notebook này.